In [ ]:
import os
os.environ["NUMBA_DISABLE_JIT"] = "1"
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
print("Pinned threads + disabled numba JIT")

from prompt import Prompt

import numpy as np
from pathlib import Path

# from gemma.ejecta.physics.jet.prompt import Prompt

def interp_logt(t_new, t, y, left=0.0, right=0.0):
    # assumes t increasing
    return np.interp(t_new, t, y, left=left, right=right)

def sample_params(rng):
    # Example priors — adjust to your science priors / bounds
    # (These should match your wrapper bounds.)
    return {
        "gamma_jet": 10**rng.uniform(np.log10(30), np.log10(1000)),
        "theta_c_jet": rng.uniform(1.0, 15.0),     # deg
        "theta_cut_jet": rng.uniform(10.0, 60.0),  # deg
        "theta_los": rng.uniform(0.0, 60.0),       # deg
        "mass_1": rng.uniform(1.1, 2.2),
        "mass_2": rng.uniform(1.0, 2.0),
        "lambda_2": rng.uniform(10.0, 2000.0),
        # if flux:
        # "luminosity_distance": rng.uniform(10.0, 500.0),  # Mpc
    }

def featurize(p):
    # Convert dict -> feature vector in a consistent order
    # (Use the same ordering everywhere.)
    gamma = np.log10(p["gamma_jet"])
    theta_c = p["theta_c_jet"] / 90.0
    theta_cut = p["theta_cut_jet"] / 90.0
    theta_los = p["theta_los"] / 90.0
    m1 = p["mass_1"]
    m2 = p["mass_2"]
    lam = np.log10(p["lambda_2"] + 1.0)
    return np.array([gamma, theta_c, theta_cut, theta_los, m1, m2, lam], dtype=np.float32)

def main():
    rng = np.random.default_rng(0)

    # Fixed emulator time grid (choose something covering your regime)
    # You can also clip to what Prompt produces, but fixed is easiest.
    t_grid = np.logspace(0, 6, 256).astype(np.float64)  # 1s .. 1e6s

    # Wrapper
    prompt = Prompt(
        components=("jet",),
        sample_gw_parameters=False,
        gw_param_mode="mass",
        j_struct="tophat",
        use_disk_mass_mapping=True,
        output="luminosity",  # or "flux"
        default_theta_los_deg=0.0,
        # if flux:
        # default_distance_mpc=40.0,
        # sample_distance=False,
    )

    N = 20000  # start small; scale later
    X = np.zeros((N, 7), dtype=np.float32)
    Y = np.zeros((N, t_grid.size), dtype=np.float32)

    # floor for log transform (pick something tiny in your chosen units)
    y_floor = 1e-40

    n_ok = 0
    i = 0
    while n_ok < N:
        i += 1
        p = sample_params(rng)

        try:
            prompt.update_model(p, dry_run=False, verbose=False)
            t_model = np.asarray(prompt.t, dtype=np.float64)
            y_model = np.asarray(prompt.total_X, dtype=np.float64)

            # interpolate onto fixed grid
            y_interp = interp_logt(t_grid, t_model, y_model, left=0.0, right=0.0)

            # log-transform target
            y_target = np.log10(y_interp + y_floor).astype(np.float32)

            X[n_ok] = featurize(p)
            Y[n_ok] = y_target
            n_ok += 1

            if n_ok % 50 == 0:
                print(f"Generated {n_ok}/{N} ok (attempts={i})")

        except Exception as e:
            # failed sample (numerical / domain issues) → skip
            continue

    out = Path("prompt_dataset.npz")
    np.savez(out, X=X, Y=Y, t_grid=t_grid, y_floor=y_floor)
    print("Saved:", out.resolve())

if __name__ == "__main__":
    main()


In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
from scripts.jet import Jet
from scripts.wind_enhanced import Wind
from scripts.magnetar_enhanced import Magnetar  # not used yet, but keeping import
import scripts.helper

import numpy as np
from scripts.helper import _int_energy_1d, _interp_1d, band_broadcast
from scripts.const import *  # expects c, etc. in CGS or consistent units

from gemma.ejecta.functions import calculate_ns_compactness
from gemma.ejecta.bns import calculate_bns_disk_mass_kruger
from gemma.utils.units_conversion import*

EV_TO_ERG = 1.602176634e-12


class Prompt:
    """
    PromptX wrapper built in GEMMA style:

    - __init__: configuration + define parameter_names/bounds only
    - update_model(params): called every likelihood evaluation with a new sample
    - generate_light(params): derive E_iso and generate prompt emission outputs
    """

    def __init__(
        self,
        components=("jet",),
        sample_gw_parameters=False,
        gw_param_mode="chirp_mass",
        j_struct=None,
        eta=1e-4,
        frac=0.1,
        phi_los_deg=0.0,
        inclination_key="theta_jn",
        # you can set a fallback if not sampling GW parameters:
        default_theta_los_deg=20.0,
        use_disk_mass_mapping=True
    ):
        self.use_disk_mass_mapping = use_disk_mass_mapping
        self.components = list(components)
        self.sample_gw_parameters = bool(sample_gw_parameters)
        self.gw_param_mode = gw_param_mode

        # Hyperparameters / fixed knobs (you can later promote them to sampled params)
        self.eta = float(eta)
        self.frac = float(frac)
        self.phi_los = np.deg2rad(phi_los_deg)
        self.inclination_key = inclination_key
        self.default_theta_los_deg = float(default_theta_los_deg)

        # Grid parameters
        self.grid_params = {"n_theta": 500, "n_phi": 100}

        # Jet structure index mapping (keep your convention)
        structs = {"tophat": 1, "gaussian": 2, "powerlaw": 3}
        if j_struct is None:
            self.j_idx = 1
        else:
            if isinstance(j_struct, str):
                if j_struct not in structs:
                    raise ValueError("Pick a valid jet profile among: tophat, gaussian, powerlaw")
                self.j_idx = structs[j_struct]
            elif isinstance(j_struct, (int, np.integer)):
                if j_struct not in (1, 2, 3):
                    raise ValueError("j_struct int must be 1 (tophat), 2 (gaussian), or 3 (powerlaw)")
                self.j_idx = int(j_struct)
            else:
                raise TypeError("j_struct must be None, a string ('gaussian'), or an int (1/2/3)")

        # ---- parameters exposed to the sampler ----
        self.parameter_names = []
        self.parameter_bounds = {}

        # Define base params (your original code referenced base_params but didn't define it)
        # This keeps things explicit and closer to how GEMMA does it.
        component_suffix_map = {"jet": "jet", "wind": "wind"}

        # Jet parameters you actually use
        jet_base_params = [
            ("gamma", (10.0, 2000.0)),
            ("theta_c", (0.2, 30.0)),      # degrees
            ("theta_cut", (0.2, 90.0)),    # degrees
        ]

        # Wind parameters you referenced
        wind_base_params = [
            ("gamma", (2.0, 300.0)),
            ("theta_cut", (0.2, 90.0)),    # degrees
            ("E_iso", (1e45, 1e55)),       # if you want to sample wind energy directly
            ("collapse", (0.0, 1.0)),      # treat as continuous; interpret >0.5 as True
            ("P_0", (0.5e-3, 50e-3)),      # seconds (example)
            ("M", (1.0, 3.0)),             # Msun
            ("B_p", (1e13, 1e16)),         # G
            ("R", (8e5, 1.6e6)),           # cm (8-16 km)
            ("eps", (1e-6, 1e-2)),
            ("kappa", (0.1, 20.0)),
            ("m_ejecta_dyn", (1e-4, 0.1)), # Msun
            ("v_ejecta_dyn", (0.05, 0.4)), # c
        ]

        # Add component parameters
        for comp in self.components:
            suffix = component_suffix_map[comp]
            if comp == "jet":
                base_params = jet_base_params
            elif comp == "wind":
                base_params = wind_base_params
            else:
                raise ValueError(f"Unknown component '{comp}', expected 'jet' and/or 'wind'")

            for base_name, bounds in base_params:
                pname = f"{base_name}_{suffix}"
                self.parameter_names.append(pname)
                self.parameter_bounds[pname] = bounds

        # If not sampling GW parameters, we must sample or set theta_los
        if not self.sample_gw_parameters:
            self.parameter_names.append("theta_los")
            self.parameter_bounds["theta_los"] = (0.0, 90.0)
        else:
            # When sampling GW params, we expect inclination to be in the global parameter dict
            # (we add it here so this wrapper can work standalone too).
            self.parameter_names.append(self.inclination_key)
            self.parameter_bounds[self.inclination_key] = (0.0, np.pi)

            # Minimal GW/EOS inputs needed for your E_iso mapping
            # (You can delete these if they come from the global sampler anyway.)
            if self.gw_param_mode == "mass":
                for name, bounds in [
                    ("mass_1", (0.8, 3.0)),
                    ("mass_2", (0.8, 3.0)),
                    ("lambda_2", (0.0, 5000.0)),
                ]:
                    self.parameter_names.append(name)
                    self.parameter_bounds[name] = bounds
            elif self.gw_param_mode == "chirp_mass":
                # NOTE: requires conversion chirp_mass+mass_ratio -> m1,m2 implemented in _extract_masses()
                for name, bounds in [
                    ("chirp_mass", (0.8, 2.0)),
                    ("mass_ratio", (0.05, 1.0)),  # q=m2/m1
                    ("lambda_2", (0.0, 5000.0)),  # simplest; or lambda_tilde if you prefer
                ]:
                    self.parameter_names.append(name)
                    self.parameter_bounds[name] = bounds
            else:
                raise ValueError("gw_param_mode must be 'mass' or 'chirp_mass'")

        # Cached state
        self.params = None
        self.jet_model = None
        self.wind_model = None
        self.total_X = None

        # Add manual jet energy if we are NOT using the ejecta/disk-mass mapping
        if ("jet" in self.components) and (not self.use_disk_mass_mapping):
            self.parameter_names.append("E_iso_jet")
            self.parameter_bounds["E_iso_jet"] = (1e45, 1e55)  # adjust prior if needed


    # -----------------------
    # Public API for sampler
    # -----------------------
    def bounds_check(self, params):
        for name in self.parameter_names:
            if name not in params:
                continue
            lo, hi = self.parameter_bounds[name]
            v = float(params[name])
            if not (lo <= v <= hi):
                raise ValueError(f"[Bound check failed] {name}={v} not in [{lo},{hi}]")

    def update_model(self, params, check_bounds=False, dry_run=True):
        print('Updating model...')
        if check_bounds:
            print('Checking bounds...')
            self.bounds_check(params)
            print('Bounds checked!')
        self.params = params
        if dry_run:
            # only compute derived quantities; no PromptX calls
            print('Starting dry run...')
            theta_los = self._get_theta_los(params)
            print('thetalos ok!')
            theta_c = np.deg2rad(float(params["theta_c_jet"]))
            print('core ok!')
            E_iso_jet = self._derive_Eiso_jet(params, theta_c)
            return {"theta_los": theta_los, "E_iso_jet": E_iso_jet}
        self.generate_light(params)

    # -----------------------
    # Core model evaluation
    # -----------------------
    def generate_light(self, params):
        """
        For this sample:
        - derive jet energetics (E_iso_jet) from disk mass mapping
        - run PromptX jet/wind models
        - cache total_X (and keep jet_model/wind_model for later extraction)
        """
        # Viewing angle
        theta_los = self._get_theta_los(params)

        # Jet
        jet_L = None
        if "jet" in self.components:
            jet_L = self.generate_jet_lc(params, theta_los)

        # Wind
        wind_L = None
        if "wind" in self.components:
            wind_L = self.generate_wind_lc(params, theta_los)

        # Combine
        if (jet_L is not None) and (wind_L is not None):
            # Make sure they are on same time grid:
            x_jet = jet_L.L_X_tot
            x_wind = wind_L.L_X_tot
            # Lw = np.interp(t, wind_L["t"], wind_L["Lx"], left=0.0, right=0.0)
            # self.total_X = {"t": t, "Lx": jet_L["Lx"] + Lw}
            self.total_X = x_jet + x_wind
            self.t = jet_L.t
        elif jet_L is not None:
            self.total_X = jet_L.L_X_tot
            self.t = jet_L.t
        elif wind_L is not None:
            self.total_X = wind_L.L_X_tot
            self.t = wind_L.engine.t
        else:
            raise RuntimeError("No components enabled")

    def generate_jet_lc(self, params, theta_los_rad):
        """
        Build and run PromptX Jet model for current params.
        Returns dict with time grid and X-ray luminosity/flux-like quantity.
        """
        # Parse jet params (degrees -> radians where needed)
        gamma_jet = float(params["gamma_jet"])
        theta_c = np.deg2rad(float(params["theta_c_jet"]))
        theta_cut = np.deg2rad(float(params["theta_cut_jet"]))
        print('Proceeding with isojet...')
        # Derive E_iso_jet from your mapping
        E_iso_jet = self._derive_Eiso_jet(params, theta_c)

        # PromptX uses eps0 in some places; common mapping is eps0 = E_iso/(4π)
        eps0 = E_iso_jet
        #eps0 = E_iso_jet / (4.0 * np.pi)

        jet_model = Jet(
            g0=gamma_jet,
            E_iso=E_iso_jet,
            eps0=eps0,
            n_theta=self.grid_params["n_theta"],
            n_phi=self.grid_params["n_phi"],
            theta_c=theta_c,
            theta_cut=theta_cut,
            jet_struct=self.j_idx,
        )
        print('Model Constructed...')

        # Ensure structure is consistent with E_iso/eps0 choice
        jet_model.define_structure(
            g0=gamma_jet,
            eps0=eps0,
            E_iso=E_iso_jet,
            jet_struct=self.j_idx,
        )

        jet_model.create_obs_grid(amati_a=0.41, amati_b=0.83)
        jet_model.observer(theta_los=float(theta_los_rad), phi_los=float(self.phi_los))
        print('Structure Constructed...')

        self.jet_model = jet_model

        return jet_model

        # t, Lx = self._extract_xray_lc(jet_model)
        # return {"t": t, "Lx": Lx}

    def generate_wind_lc(self, params, theta_los_rad):
        """
        Build and run PromptX Wind model for current params.
        Returns dict with time grid and X-ray luminosity/flux-like quantity.
        """
        gamma_wind = float(params["gamma_wind"])
        theta_cut = np.deg2rad(float(params["theta_cut_wind"]))

        wind = Wind(
            g0=gamma_wind,
            n_theta=self.grid_params["n_theta"],
            n_phi=self.grid_params["n_phi"],
            theta_cut=theta_cut,
        )
        wind.observer(theta_los=float(theta_los_rad), phi_los=float(self.phi_los))

        # If Wind needs extra init (E_iso, collapse, magnetar params) do it here.
        # Your original script stored them, so keep them available:
        wind.E_iso = float(params["E_iso_wind"])
        wind.collapse = bool(float(params["collapse_wind"]) > 0.5)

        self.wind_model = wind

        return wind

    # -----------------------
    # Helpers
    # -----------------------
    def _get_theta_los(self, params):
        if self.sample_gw_parameters:
            if self.inclination_key not in params:
                raise KeyError(f"Missing GW inclination key '{self.inclination_key}' in params")
            return float(params[self.inclination_key])  # already radians
        # otherwise sample or use default
        if "theta_los" in params:
            return np.deg2rad(float(params["theta_los"]))
        return np.deg2rad(self.default_theta_los_deg)

    def _extract_masses(self, params):
        """
        Returns (m1, m2) in Msun.
        Supports gw_param_mode 'mass' directly.
        For 'chirp_mass', you MUST plug in bilby conversion (or provide mass_1/mass_2 globally).
        """
        if "mass_1" in params and "mass_2" in params:
            return float(params["mass_1"]), float(params["mass_2"])

        if self.gw_param_mode == "mass":
            raise KeyError("gw_param_mode='mass' requires mass_1 and mass_2 in params")

        if self.gw_param_mode == "chirp_mass":
            # If you already have bilby conversion available in your environment,
            # you should import it and do the proper conversion here.
            # Example:
            # from gemma.utils import bilby_conversion as bilby_conv
            # m1, m2 = bilby_conv.chirp_mass_and_mass_ratio_to_component_masses(params["chirp_mass"], params["mass_ratio"])
            raise ValueError(
                "gw_param_mode='chirp_mass' requires chirp_mass+mass_ratio conversion. "
                "Either provide mass_1/mass_2 directly in params, or implement bilby conversion here."
            )

        raise ValueError(f"Unknown gw_param_mode: {self.gw_param_mode}")

    def _derive_Eiso_jet(self, params, theta_c_rad):
        """
        Your intent: compute disk mass from Kruger fit, apply efficiency, then convert to E_iso.

        This version:
          E_k = (1-frac)*(1-xi)*M_disk*eta*c^2
          E_iso ≈ 2 E_k / theta_c^2   (small-angle Gaussian/powerlaw normalization)
        """
        # Need mass_1/mass_2 and lambda_2 at minimum
        if not self.use_disk_mass_mapping:
            if "E_iso_jet" not in params:
                raise KeyError("use_disk_mass_mapping=False but E_iso_jet not provided")
            return float(params["E_iso_jet"])

        m1, m2 = self._extract_masses(params)
        if "lambda_2" not in params:
            raise KeyError("lambda_2 missing: needed for compactness -> disk mass -> E_iso_jet")
        lambda_2 = float(params["lambda_2"])

        # Compactness and disk mass (Kruger)
        C2 = float(calculate_ns_compactness(lambda_2, True, systematics_fraction=0.0))
        print('Compactness Calculated...')
        m_disk = float(calculate_bns_disk_mass_kruger(m2, C2))  # assumed Msun

        # Your xi(q) expression (q interpreted as m1/m2)
        q = max(1e-6, m1 / m2)
        xi = 0.18 + 0.11 / (1.0 + np.exp(1.5 * (q - 3.0)))

        # Energy budget (units depend on scripts.const; assume c in cm/s and M_disk in Msun)
        # Convert Msun -> grams if cgs constants expect that:
        E_k = (1.0 - self.frac) * (1.0 - xi) * (m_disk*M_sun_cgs) * self.eta * (c_cgs ** 2)

        # Map to E_iso
        denom = max(1e-12, 1.0 - np.cos(theta_c_rad))
        E_iso = 2.0 * E_k / denom
        return float(E_iso)

    @staticmethod
    def _extract_xray_lc(obj):
        """
        Try to extract (t, Lx) arrays from PromptX Jet/Wind objects.
        Adjust attribute names here once you confirm PromptX outputs.

        Returns numpy arrays.
        """
        t = None
        for k in ("t", "t_obs", "t_grid", "_t"):
            if hasattr(obj, k):
                t = getattr(obj, k)
                break

        Lx = None
        for k in ("L_X_tot", "Lx", "lx", "F_X_tot", "Fx"):
            if hasattr(obj, k):
                Lx = getattr(obj, k)
                break

        if t is None or Lx is None:
            # Help you debug quickly
            candidates = [k for k in dir(obj) if ("X" in k or "t" in k or "lc" in k or "flux" in k or "lum" in k)]
            raise AttributeError(
                f"Could not extract X-ray lightcurve from {type(obj)}. "
                f"Candidate attributes: {candidates}"
            )

        return np.asarray(t, dtype=float), np.asarray(Lx, dtype=float)

    def check_print(self):
        print('Object OK!')